# colab_run_models_v1 — VPD 1시간 뒤 예측 모델 학습·평가 (전체 파이프라인)

이 노트북은 지금까지 Drive에 모아둔 4개 출처의 스마트팜 환경데이터를 하나로 합친 뒤,
**RandomForest / LightGBM / XGBoost / GRU(직접 구현) / GRU+LightGBM 앙상블** 5개 모델로
"1시간 뒤 VPD(수증기압차)"를 예측하고, **persistence 기준선**(그냥 현재 값을 그대로
쓰는 방법)과 비교합니다.

여기서 나오는 숫자들이 VPD_단기예측_논문.docx의 표1·표2·본문에 적힌 숫자의 근거입니다.

In [2]:
# =============================================================================
# colab_run_models_v1.py
#
# 목적: colab_trim_farms_v2.py 등으로 만들어둔 "1시간 단위 점샘플링" 데이터들을
#       모두 모아서, 실제 VPD 단기예측 모델(RandomForest / LightGBM / XGBoost /
#       GRU / GRU+LightGBM 앙상블)을 학습·평가하고, persistence 기준선과 비교한다.
#
#       이 스크립트는 논문 2("VPD 단기예측") 표1·표2에 실린 숫자를 그대로 재현하는
#       코드다. 즉 여기서 나오는 RMSE/MAE/R² 숫자가 논문에 적힌 숫자와 (거의) 같아야
#       정상이다. (완전히 똑같지 않을 수 있는 이유: GRU는 랜덤 초기화가 들어가는
#       신경망이라 실행 환경(GPU/CPU, 라이브러리 버전)에 따라 아주 미세하게 값이
#       흔들릴 수 있다. RandomForest/LightGBM/XGBoost/persistence는 완전히 결정적
#       이라 항상 같은 값이 나온다.)
#
# 전체 흐름 (Part A: 데이터 준비 -> Part B: 모델 학습/평가):
#   [Part A] 4개 출처의 데이터를 모아서 "10개 온실(source)"의 시간단위 데이터로 통합
#     1) aihub_farms_hourly_pointsample_v1.csv : AI Hub 1·4·5·6번 농가 (이미 점샘플링 완료본, Drive에 있음)
#     2) aihub_farm2_hourly_pointsample.csv    : AI Hub 2번 농가 (이미 점샘플링 완료본, Drive에 있음)
#     3) 3번농가(프리바) 원본 CSV               : 이 스크립트 안에서 직접 점샘플링(1~5분 -> 1시간)
#     4) 24_환경정보.xlsx (2024 경진대회)         : 이미 시간 단위라 그대로 사용, 시트 4개 = 온실 4개
#     -> 4곳에서 온 데이터를 하나의 표로 합치고 VPD(수증기압차)를 계산한다.
#
#   [Part B] 통합된 데이터로 "1시간 뒤 VPD"를 예측하는 모델을 학습·평가
#     - 실험1: 각 온실마다 시간순으로 앞 80%는 학습, 뒤 20%는 테스트 (실제 예보 상황 재현)
#     - 실험2: Leave-One-Source-Out(LOSO) — 한 온실을 통째로 빼고 학습한 뒤 그 온실에 적용
#              (완전히 처음 보는 온실에서도 모델이 통하는지 확인)
#     - 두 실험 모두 persistence 기준선(그냥 "1시간 전 값을 그대로 쓴다")과 비교한다.
#
# =============================================================================


# #############################################################################
# PART A. 데이터 준비: 4개 출처를 모아 "10개 온실" 통합 데이터셋 만들기
# #############################################################################

# optax(GRU 최적화에 필요)와 openpyxl(엑셀 읽기)는 Colab 기본 설치에 없을 수 있어 먼저 설치한다.

!pip install -q optax openpyxl

import numpy as np
import pandas as pd
import glob
import openpyxl
from google.colab import drive

drive.mount('/content/drive')

# -----------------------------------------------------------------------------
# A-1) 이미 만들어둔 "점샘플링 완료" 파일 2개를 Drive에서 찾아 불러온다
#      (colab_trim_farms_v2.py / colab_trim_farm2_v2.py 를 이미 실행해서 Drive에
#       저장해둔 결과물. 못 찾으면 그 두 스크립트를 먼저 실행해야 한다.)
# -----------------------------------------------------------------------------
def find_one(filename):
    matches = glob.glob(f'/content/drive/**/{filename}', recursive=True)
    if not matches:
        raise FileNotFoundError(
            f'{filename} 을 Drive에서 찾지 못했습니다. '
            f'colab_trim_farms_v2.py / colab_trim_farm2_v2.py 를 먼저 실행해서 '
            f'이 파일을 만들어 두세요.'
        )
    return matches[0]

# 1·4·5·6번 농가 (그린CS 장비, 이미 1시간 단위 점샘플링 완료됨)
path_1456 = find_one('aihub_farms_hourly_pointsample_v1.csv')
a1456 = pd.read_csv(path_1456, parse_dates=['date'])
a1456['source'] = 'aihub_farm' + a1456['farm_id'].astype(str)   # 예: farm_id=1 -> 'aihub_farm1'
a1456 = a1456.drop(columns=['farm_id'])
print('[1/4] aihub 1·4·5·6번 농가 로드:', a1456.shape, dict(a1456['source'].value_counts()))

# 2번 농가 (프리바 장비, 컬럼 구조가 달라 별도 파일로 관리됨, 이미 점샘플링 완료됨)
path_farm2 = find_one('aihub_farm2_hourly_pointsample.csv')
a2 = pd.read_csv(path_farm2, parse_dates=['date'])
a2['source'] = 'aihub_farm2'
a2 = a2.drop(columns=['farm_id'])
print('[2/4] aihub 2번 농가 로드:', a2.shape)

# -----------------------------------------------------------------------------
# A-2) 3번 농가(프리바, 2023년, 1~5분 간격 원시데이터)는 아직 점샘플링을 해둔
#      적이 없으므로, 여기서 직접 1시간 단위로 축약한다.
#      -> colab_trim_farms_v2.py와 완전히 같은 방식: .resample('1h').first()
# -----------------------------------------------------------------------------
path_farm3_raw = find_one('토마토_환경데이터_3번농가(프리바).csv')
f3 = pd.read_csv(path_farm3_raw, low_memory=False)

# 프리바 장비는 컬럼명이 그린CS와 다르다 (예: 'meas grh temp:CMP (°C)' 처럼 영문+단위 표기)
f3 = f3.rename(columns={
    'datetime': 'date',
    'meas grh temp:CMP (°C)': 'temp',
    'meas RH:CMP (%)': 'humidity',
    'meas CO2:CMP (ppm)': 'co2',
})
f3['date'] = pd.to_datetime(f3['date'], errors='coerce')
f3 = f3[['date', 'temp', 'humidity', 'co2']].dropna(subset=['date'])
for c in ['temp', 'humidity', 'co2']:
    f3[c] = pd.to_numeric(f3[c], errors='coerce')

# *** 핵심: 원본 1~5분 간격 -> 1시간 단위 "첫 실측값"만 사용 (평균 아님, 이유는
#     colab_trim_farms_v2.py 상단 설명 참고 — 평균을 쓰면 노이즈가 인위적으로
#     매끈해져서, 애초에 평균화 없이 수집된 2024년 경진대회 데이터와 조건이 달라진다) ***
f3 = f3.set_index('date').sort_index().resample('1h').first()
f3 = f3.dropna(subset=['temp', 'humidity']).reset_index()
f3['source'] = 'farm3_2023'
print('[3/4] farm3(2023) 원본 -> 점샘플링:', f3.shape)

# -----------------------------------------------------------------------------
# A-3) 2024년 경진대회 데이터(24_환경정보.xlsx)는 애초에 1시간 단위로만 수집되어
#      있어서 별도 리샘플링이 필요 없다. 엑셀 시트 1개 = 온실 1개(총 4개 시트).
# -----------------------------------------------------------------------------
path_gh24 = find_one('24_환경정보.xlsx')
wb = openpyxl.load_workbook(path_gh24, read_only=True)
frames24 = []
for sheet_name in wb.sheetnames:   # 시트 이름 예: '1온실', '2온실', '3온실', '4온실'
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    header, data = rows[0], rows[1:]
    df = pd.DataFrame(data, columns=header)
    df = df.rename(columns={'측정일시': 'date', '내부온도': 'temp', '내부습도': 'humidity', '내부CO2': 'co2'})
    df = df[['date', 'temp', 'humidity', 'co2']].copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    for c in ['temp', 'humidity', 'co2']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['date', 'temp', 'humidity'])
    df['source'] = f'gh24_{sheet_name}'   # 예: 'gh24_1온실'
    frames24.append(df)
    print(f'  {sheet_name}: {df.shape}')
df24 = pd.concat(frames24, ignore_index=True)
print('[4/4] 2024 경진대회 4개 온실 로드:', df24.shape)

# -----------------------------------------------------------------------------
# A-4) 4곳의 데이터를 하나로 합치고, VPD(수증기압차)를 계산한다.
#      VPD 공식은 생육예측 논문(논문1)과 동일 (Tetens 근사식 기반):
#        포화수증기압 SVP = 0.6108 * exp(17.27*T / (T+237.3))   [T: 섭씨온도]
#        실제수증기압 AVP = SVP * (상대습도RH / 100)
#        수증기압차   VPD = SVP - AVP
# -----------------------------------------------------------------------------
combined = pd.concat(
    [f3[['date', 'temp', 'humidity', 'co2', 'source']], a1456, a2, df24],
    ignore_index=True,
)
combined = combined.sort_values(['source', 'date']).reset_index(drop=True)

T = combined['temp']
RH = combined['humidity']
SVP = 0.6108 * np.exp(17.27 * T / (T + 237.3))
AVP = SVP * (RH / 100.0)
combined['VPD'] = SVP - AVP

print('\n=== 통합 데이터셋 완성 ===')
print('전체 shape:', combined.shape, '(10개 온실 합산)')
print(combined['source'].value_counts())

combined.to_csv('/content/drive/MyDrive/combined_hourly_pointsample.csv', index=False)
print('\n저장 완료: /content/drive/MyDrive/combined_hourly_pointsample.csv')


# #############################################################################
# PART B. 모델 학습 및 평가 (RandomForest / LightGBM / XGBoost / GRU / 앙상블)
#         + persistence 기준선 비교
# #############################################################################

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
import jax, jax.numpy as jnp
import optax

RNG = 42
np.random.seed(RNG)

df = combined.sort_values(['source', 'date']).reset_index(drop=True)
# light(일사량) 컬럼은 aihub_farm2에는 없어서, "모든 온실에 공통으로 있는" 4개 변수만 피처로 사용
BASE = ['temp', 'humidity', 'co2', 'VPD']

# ---------- lag 피처 (t-2, t-1, t) 만들기 + "진짜 연속된 1시간 간격"만 남기기 ----------
# 예보 대상: target_VPD = 그 온실의 "1시간 뒤" VPD 값
for lag in [1, 2]:
    for c in BASE:
        df[f'{c}_lag{lag}'] = df.groupby('source')[c].shift(lag)
df['target_VPD'] = df.groupby('source')['VPD'].shift(-1)  # t+1 예보 대상

# 온실 내에서 앞/뒤 행과의 시간 간격(시간)을 계산 -> 결측/누락 구간을 걸러내기 위함
df['dt_back'] = df.groupby('source')['date'].diff().dt.total_seconds() / 3600
df['dt_fwd'] = -df.groupby('source')['date'].diff(-1).dt.total_seconds() / 3600

# t-2, t-1, t, t+1 이 전부 정확히 1시간 간격으로 끊김 없이 이어진 행만 사용한다.
# (중간에 결측이 있는데 억지로 보간해서 채우면, "진짜 1시간 뒤 예보"가 아니게 되어
#  모델 성능이 과장될 수 있다 — 그래서 엄격하게 연속 구간만 걸러낸다.)
mask = (df['dt_back'] == 1) & (df['dt_fwd'] == 1) & df[[f'{c}_lag2' for c in BASE]].notna().all(axis=1)
df_clean = df[mask].dropna(subset=['target_VPD']).reset_index(drop=True)
print('연속 1시간 간격 유효 샘플:', len(df_clean), '/', len(df), dict(df_clean['source'].value_counts()))

# 트리 계열 모델(RF/LGBM/XGB)용: (t-2, t-1, t) 3개 시점 x 4개 변수 = 12개 + 현재 t의 4개 = 15개 피처를 1줄로 펼침
FLAT_FEATURES = BASE + [f'{c}_lag1' for c in BASE] + [f'{c}_lag2' for c in BASE]  # 총 15개 피처
X_flat = df_clean[FLAT_FEATURES].values
y = df_clean['target_VPD'].values
src = df_clean['source'].values

# *** persistence 기준선 예측값 = "그냥 현재(t) VPD를 그대로 1시간 뒤 예측값으로 쓴다" ***
# df_clean['VPD']가 바로 그 t 시점의 실측 VPD이므로, 이걸 그대로 예측값처럼 사용해서
# y(=target_VPD, t+1 실측값)와 비교하면 persistence의 RMSE/MAE/R²를 구할 수 있다.
persist_all = df_clean['VPD'].values

# GRU(순환신경망)용: 시계열 형태 그대로 (샘플 수, 3개 시점, 4개 변수) 3차원 배열
X_seq = np.stack([
    df_clean[[f'{c}_lag2' for c in BASE]].values,   # t-2 시점
    df_clean[[f'{c}_lag1' for c in BASE]].values,   # t-1 시점
    df_clean[BASE].values,                           # t 시점
], axis=1)
print('X_flat', X_flat.shape, 'X_seq', X_seq.shape)


def run_suite(Xf_tr, Xf_te, Xs_tr, Xs_te, y_tr, y_te, persist_te, label):
    """학습 데이터로 5개 모델(+persistence 기준선)을 학습/계산하고, 테스트 세트에서
    RMSE/MAE/R²를 출력한다. persist_te: 테스트 세트의 persistence 예측값(=현재 VPD)."""
    results = {}

    # --- persistence 기준선: 학습이 필요 없다. "현재 값 = 1시간 뒤 예측값" ---
    results['Persistence'] = persist_te

    # --- RandomForest ---
    rf = RandomForestRegressor(n_estimators=300, random_state=RNG, n_jobs=-1)
    rf.fit(Xf_tr, y_tr)
    results['RandomForest'] = rf.predict(Xf_te)

    # --- LightGBM ---
    lgbm = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=RNG, verbose=-1)
    lgbm.fit(Xf_tr, y_tr)
    results['LightGBM'] = lgbm.predict(Xf_te)

    # --- XGBoost ---
    xgbm = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=RNG, verbosity=0)
    xgbm.fit(Xf_tr, y_tr)
    results['XGBoost'] = xgbm.predict(Xf_te)

    # --- GRU (JAX로 직접 구현한 순환신경망) ---
    # 입력 스케일을 표준화(평균0, 분산1)해야 신경망 학습이 안정적으로 된다.
    scaler_x = StandardScaler().fit(Xs_tr.reshape(-1, Xs_tr.shape[-1]))
    def scale_seq(a):
        shp = a.shape
        return scaler_x.transform(a.reshape(-1, shp[-1])).reshape(shp)
    Xs_tr_n = scale_seq(Xs_tr)
    Xs_te_n = scale_seq(Xs_te)
    y_mean, y_std = y_tr.mean(), y_tr.std()
    y_tr_n = (y_tr - y_mean) / y_std

    D = Xs_tr.shape[-1]   # 피처 개수(4)
    H = 12                # GRU 은닉상태 크기

    def init_params(key):
        k = jax.random.split(key, 8)
        s = 0.3
        return {
            'Wz': jax.random.normal(k[0], (D + H, H)) * s, 'bz': jnp.zeros(H),
            'Wr': jax.random.normal(k[1], (D + H, H)) * s, 'br': jnp.zeros(H),
            'Wh': jax.random.normal(k[2], (D + H, H)) * s, 'bh': jnp.zeros(H),
            'Wo': jax.random.normal(k[3], (H, 1)) * s, 'bo': jnp.zeros(1),
        }

    def gru_forward(params, x_seq):
        h = jnp.zeros(H)
        def step(h, x):
            zx = jnp.concatenate([x, h])
            z = jax.nn.sigmoid(zx @ params['Wz'] + params['bz'])       # update gate
            r = jax.nn.sigmoid(zx @ params['Wr'] + params['br'])       # reset gate
            rx = jnp.concatenate([x, r * h])
            hh = jnp.tanh(rx @ params['Wh'] + params['bh'])            # 후보 은닉상태
            h_new = (1 - z) * h + z * hh
            return h_new, h_new
        h, _ = jax.lax.scan(step, h, x_seq)
        out = h @ params['Wo'] + params['bo']
        return out[0]

    batched_forward = jax.vmap(gru_forward, in_axes=(None, 0))

    def loss_fn(params, xb, yb, l2=1e-3):
        pred = batched_forward(params, xb)
        mse = jnp.mean((pred - yb) ** 2)
        reg = l2 * sum(jnp.sum(v ** 2) for k, v in params.items() if k.startswith('W'))
        return mse + reg

    # 랜덤 초기화에 따른 편차를 줄이기 위해 3개의 다른 시드로 학습해보고,
    # 그중 학습 데이터 RMSE가 가장 낮은 모델을 채택한다.
    best_params, best_val = None, 1e9
    n_tr = len(Xs_tr_n)
    batch_size = min(512, n_tr)
    for seed in range(3):
        params = init_params(jax.random.PRNGKey(seed))
        opt = optax.adam(5e-3)
        opt_state = opt.init(params)

        @jax.jit
        def train_step(params, opt_state, xb, yb):
            loss, grads = jax.value_and_grad(loss_fn)(params, xb, yb)
            updates, opt_state = opt.update(grads, opt_state)
            params = optax.apply_updates(params, updates)
            return params, opt_state, loss

        rng = np.random.RandomState(seed)
        for epoch in range(60):
            perm = rng.permutation(n_tr)
            for start in range(0, n_tr, batch_size):
                bidx = perm[start:start + batch_size]
                params, opt_state, loss = train_step(
                    params, opt_state, jnp.array(Xs_tr_n[bidx]), jnp.array(y_tr_n[bidx]))
        tr_pred_n = np.array(batched_forward(params, jnp.array(Xs_tr_n)))
        tr_rmse = np.sqrt(mean_squared_error(y_tr_n, tr_pred_n))
        if tr_rmse < best_val:
            best_val, best_params = tr_rmse, params

    params = best_params
    gru_pred_n = np.array(batched_forward(params, jnp.array(Xs_te_n)))
    results['GRU'] = gru_pred_n * y_std + y_mean   # 정규화 되돌리기

    # --- GRU + LightGBM 앙상블 ---
    # 학습셋의 마지막 10%를 검증용으로 떼어내, GRU 예측과 LightGBM 예측을
    # 몇 대 몇 비율로 섞을지(w) 검증 RMSE가 가장 낮아지는 지점을 찾는다.
    n_val = max(1, int(0.1 * len(Xf_tr)))
    Xf_tr2, Xf_val = Xf_tr[:-n_val], Xf_tr[-n_val:]
    Xs_tr2_n, Xs_val_n = Xs_tr_n[:-n_val], Xs_tr_n[-n_val:]
    y_tr2, y_val = y_tr[:-n_val], y_tr[-n_val:]

    lgb_val_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=RNG, verbose=-1)
    lgb_val_model.fit(Xf_tr2, y_tr2)
    val_lgb_pred = lgb_val_model.predict(Xf_val)
    val_gru_pred = np.array(batched_forward(params, jnp.array(Xs_val_n))) * y_std + y_mean

    best_w, best_rmse = 0.5, 1e9
    for w in np.linspace(0, 1, 21):
        blend = w * val_gru_pred + (1 - w) * val_lgb_pred
        rmse = np.sqrt(mean_squared_error(y_val, blend))
        if rmse < best_rmse:
            best_rmse, best_w = rmse, w
    results['GRU+LightGBM'] = best_w * results['GRU'] + (1 - best_w) * results['LightGBM']
    print(f'  [{label}] 앙상블 가중치 w(GRU)={best_w:.2f}, 검증 RMSE={best_rmse:.4f}')

    # --- 결과 출력 (Persistence를 포함해서 표 형태로) ---
    print(f'\n=== {label} (train {len(y_tr)} / test {len(y_te)}) ===')
    print(f'{"모델":<16}{"RMSE":>10}{"MAE":>10}{"R2":>10}')
    for name, pred in results.items():
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        mae = mean_absolute_error(y_te, pred)
        r2 = r2_score(y_te, pred)
        print(f'{name:<16}{rmse:>10.4f}{mae:>10.4f}{r2:>10.4f}')
    return results


# =========================================================================
# 실험 1: 소스 내부 시간순 분할 (각 온실마다 마지막 20% 시점을 테스트로) — 실제 예보 상황 재현
# =========================================================================
train_mask = np.zeros(len(df_clean), dtype=bool)
for s in np.unique(src):
    idx = np.where(src == s)[0]
    cut = int(len(idx) * 0.8)
    train_mask[idx[:cut]] = True
test_mask = ~train_mask

print('\n' + '#' * 90)
print('실험 1: 소스 내 시간순 분할 (각 온실 마지막 20% 시점 = 테스트)')
print('#' * 90)
r1 = run_suite(
    X_flat[train_mask], X_flat[test_mask],
    X_seq[train_mask], X_seq[test_mask],
    y[train_mask], y[test_mask],
    persist_all[test_mask],
    '시간순 분할',
)

# =========================================================================
# 실험 2: 온실 완전 배제(leave-one-source-out) — 학습에 전혀 없던 온실에 대한 일반화
# =========================================================================
print('\n' + '#' * 90)
print('실험 2: Leave-one-source-out (미지의 온실 일반화 검증)')
print('#' * 90)
loso_results = {}
for held_out in np.unique(src):
    tr = src != held_out
    te = src == held_out
    if te.sum() < 30:
        continue
    res = run_suite(
        X_flat[tr], X_flat[te], X_seq[tr], X_seq[te],
        y[tr], y[te], persist_all[te],
        f'LOSO holdout={held_out}',
    )
    loso_results[held_out] = res

# =========================================================================
# 요약: 각 온실별 "최고 모델 vs persistence" 개선율 + 그룹(AI Hub vs 2024 경진대회) 비교
# (compare_pointsample.py 와 동일한 집계 로직 — 논문 표2, Ⅱ장 결과 문단의 근거)
# =========================================================================
print('\n' + '#' * 90)
print('요약: 온실별 최고 모델 vs Persistence 개선율')
print('#' * 90)
print(f"{'source':<16}{'best_model':<16}{'best_rmse':>10}{'persist_rmse':>13}{'improve%':>10}")
GH24 = {'gh24_1온실', 'gh24_2온실', 'gh24_3온실', 'gh24_4온실'}
groupA, groupB = [], []
for held_out, res in loso_results.items():
    persist_rmse = np.sqrt(mean_squared_error(y[src == held_out], res['Persistence']))
    model_names = [k for k in res if k != 'Persistence']
    best_model = min(model_names, key=lambda k: np.sqrt(mean_squared_error(y[src == held_out], res[k])))
    best_rmse = np.sqrt(mean_squared_error(y[src == held_out], res[best_model]))
    improve = (persist_rmse - best_rmse) / persist_rmse * 100
    print(f"{held_out:<16}{best_model:<16}{best_rmse:>10.4f}{persist_rmse:>13.4f}{improve:>9.1f}%")
    (groupB if held_out in GH24 else groupA).append(improve)

import statistics as st
print()
print(f"Group A (AI Hub 농가, n={len(groupA)}): mean={st.mean(groupA):.1f}%, "
      f"range=[{min(groupA):.1f}%, {max(groupA):.1f}%], all_beat_persistence={all(x > 0 for x in groupA)}")
print(f"Group B (2024 경진대회 온실, n={len(groupB)}): mean={st.mean(groupB):.1f}%, "
      f"range=[{min(groupB):.1f}%, {max(groupB):.1f}%], all_lose_to_persistence={all(x < 0 for x in groupB)}")


Mounted at /content/drive
[1/4] aihub 1·4·5·6번 농가 로드: (1030, 5) {'aihub_farm1': np.int64(455), 'aihub_farm6': np.int64(441), 'aihub_farm5': np.int64(91), 'aihub_farm4': np.int64(43)}
[2/4] aihub 2번 농가 로드: (2304, 5)
[3/4] farm3(2023) 원본 -> 점샘플링: (2014, 5)
  1온실: (3061, 5)
  2온실: (3061, 5)
  3온실: (3085, 5)
  4온실: (3085, 5)
[4/4] 2024 경진대회 4개 온실 로드: (12292, 5)

=== 통합 데이터셋 완성 ===
전체 shape: (17640, 6) (10개 온실 합산)
source
gh24_4온실       3085
gh24_3온실       3085
gh24_1온실       3061
gh24_2온실       3061
aihub_farm2    2304
farm3_2023     2014
aihub_farm1     455
aihub_farm6     441
aihub_farm5      91
aihub_farm4      43
Name: count, dtype: int64

저장 완료: /content/drive/MyDrive/combined_hourly_pointsample.csv
연속 1시간 간격 유효 샘플: 17545 / 17640 {'gh24_4온실': np.int64(3076), 'gh24_3온실': np.int64(3076), 'gh24_1온실': np.int64(3052), 'gh24_2온실': np.int64(3052), 'aihub_farm2': np.int64(2301), 'farm3_2023': np.int64(2011), 'aihub_farm1': np.int64(432), 'aihub_farm6': np.int64(426), 'aihub_farm5': np.int64(79

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  [시간순 분할] 앙상블 가중치 w(GRU)=0.60, 검증 RMSE=0.0528

=== 시간순 분할 (train 14030 / test 3515) ===
모델                    RMSE       MAE        R2
Persistence         0.0550    0.0232    0.9705
RandomForest        0.0567    0.0267    0.9686
LightGBM            0.0503    0.0240    0.9753
XGBoost             0.0597    0.0283    0.9652
GRU                 0.0541    0.0286    0.9714
GRU+LightGBM        0.0509    0.0254    0.9747

##########################################################################################
실험 2: Leave-one-source-out (미지의 온실 일반화 검증)
##########################################################################################


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  [LOSO holdout=aihub_farm1] 앙상블 가중치 w(GRU)=0.50, 검증 RMSE=0.0344

=== LOSO holdout=aihub_farm1 (train 17113 / test 432) ===
모델                    RMSE       MAE        R2
Persistence         0.0982    0.0431    0.7932
RandomForest        0.0903    0.0477    0.8251
LightGBM            0.0905    0.0501    0.8243
XGBoost             0.1209    0.0731    0.6865
GRU                 0.0899    0.0449    0.8265
GRU+LightGBM        0.0870    0.0446    0.8375


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


... (13 outputs left out)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  [LOSO holdout=gh24_3온실] 앙상블 가중치 w(GRU)=0.45, 검증 RMSE=0.0323

=== LOSO holdout=gh24_3온실 (train 14469 / test 3076) ===
모델                    RMSE       MAE        R2
Persistence         0.1037    0.0301    0.9778
RandomForest        0.1308    0.0520    0.9647
LightGBM            0.1156    0.0434    0.9724
XGBoost             0.1241    0.0496    0.9682
GRU                 0.1154    0.0441    0.9725
GRU+LightGBM        0.1127    0.0404    0.9738


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  [LOSO holdout=gh24_4온실] 앙상블 가중치 w(GRU)=0.60, 검증 RMSE=0.0312

=== LOSO holdout=gh24_4온실 (train 14469 / test 3076) ===
모델                    RMSE       MAE        R2
Persistence         0.0711    0.0237    0.9839
RandomForest        0.0803    0.0330    0.9795
LightGBM            0.0742    0.0311    0.9825
XGBoost             0.0788    0.0319    0.9803
GRU                 0.0738    0.0298    0.9827
GRU+LightGBM        0.0718    0.0277    0.9836

##########################################################################################
요약: 온실별 최고 모델 vs Persistence 개선율
##########################################################################################
source          best_model       best_rmse persist_rmse  improve%
aihub_farm1     GRU+LightGBM        0.0870       0.0982     11.4%
aihub_farm2     GRU+LightGBM        0.1073       0.1243     13.7%
aihub_farm4     RandomForest        0.0799       0.0927     13.8%
aihub_farm5     GRU+LightGBM        0.0940       0.1236     24.0%
aihub_